# 01 | Huấn luyện mô hình V4: MIMIC + AFDB, Pooled LOSO 60 bệnh nhân

**Mục đích.** Tái tạo model V4 đang chạy trên AI-Service (`models/final/healthsense_afib_pipeline.pkl`): gộp hai bộ dữ liệu, cân bằng nguồn, đánh giá bằng Leave-One-Subject-Out trên 60 bệnh nhân, chọn model tốt nhất theo ROC-AUC rồi huấn luyện lại trên toàn bộ dữ liệu.

Notebook này là bản đọc được của `scripts/run_final_model.py`; mọi phép tính đặc trưng nằm trong `src/healthsense_ml`, notebook chỉ lo phần huấn luyện.

| | |
|---|---|
| Đầu vào | `data/features/v4/mimic_features_v4.csv` (35 bệnh nhân, PPG) và `data/features/v5/afdb_features_v5_afib_only.csv` (AFDB, ECG) |
| Đầu ra | `models/final/healthsense_afib_pipeline.pkl`, `model_card.json`, `pooled_loso_results.csv` |
| Thời gian | LOSO 60 fold × 3 model: vài phút trên máy tính cá nhân |
| Đặc trưng | 13 đặc trưng HRV (`config.CORE_FEATURES`), đã bỏ nhóm LF vì cửa sổ 30 giây |

Đặt `RUN_POOLED_LOSO = False` nếu chỉ muốn huấn luyện model cuối (mặc định chọn XGBoost, là model thắng ở lần chạy chuẩn).

In [1]:
# Tìm gốc repo (chạy được từ notebooks/training/ hoặc từ gốc) và nạp thư viện src/healthsense_ml
import os, sys, json, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_ml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Không tìm thấy gốc repo (thư mục chứa src/healthsense_ml)")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

import numpy as np
import pandas as pd
import joblib
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

Repo root: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab


In [2]:
RUN_POOLED_LOSO = True   # False: bỏ qua đánh giá, huấn luyện thẳng model cuối

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix
import xgboost as xgb

from healthsense_ml import config, afdb
from healthsense_ml.feature_extraction import load_features as load_mimic
from healthsense_ml.training import iqr_train_mask

OUTPUT_DIR = Path(config.MODELS_DIR) / "final"
FEATURE_COLS = config.CORE_FEATURES
print(len(FEATURE_COLS), "đặc trưng:", FEATURE_COLS)

13 đặc trưng: ['HR_mean', 'Mean_NN', 'SDNN', 'RMSSD', 'NN50', 'pNN50', 'CV', 'HF', 'Total_Power', 'HF_norm', 'SD1', 'SD2', 'SampEn']


## 1. Nạp và gộp dữ liệu

MIMIC PERform AF cho 4 130 cửa sổ PPG của 35 bệnh nhân; AFDB cho khoảng 29 000 cửa sổ từ ECG. Nếu trộn thô, AFDB sẽ đè MIMIC theo tỉ lệ 7:1, nên mỗi cửa sổ mang một **trọng số nguồn** sao cho hai dataset đóng góp tổng trọng số bằng nhau.

In [3]:
mimic = load_mimic(); mimic["source"] = "mimic"
af = afdb.load_features(); af["source"] = "afdb"
pooled = pd.concat([mimic, af], ignore_index=True)

print(f"{len(pooled)} cửa sổ | {pooled.record_id.nunique()} bệnh nhân "
      f"(MIMIC {mimic.record_id.nunique()}, AFDB {af.record_id.nunique()})")
pooled.groupby(["source", "status"]).size().unstack(fill_value=0)

32864 cửa sổ | 60 bệnh nhân (MIMIC 35, AFDB 25)


status,0,1
source,,
afdb,17713,11021
mimic,1888,2242


In [4]:
def source_weights(sources):
    """Mỗi dataset đóng góp tổng trọng số bằng nhau; chuẩn hoá để trung bình = 1."""
    sources = np.asarray(sources); n = len(sources)
    w = np.empty(n, dtype=float)
    for s in np.unique(sources):
        m = sources == s
        w[m] = n / (len(np.unique(sources)) * m.sum())
    return w

def build_models():
    rs = config.RANDOM_STATE
    return {
        "Logistic Regression": Pipeline([("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, max_iter=2000, random_state=rs))]),
        "Random Forest": Pipeline([("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(n_estimators=150, max_depth=10, random_state=rs, n_jobs=-1))]),
        "XGBoost": Pipeline([("scaler", StandardScaler()),
            ("clf", xgb.XGBClassifier(max_depth=4, learning_rate=0.05, n_estimators=150,
                                      eval_metric="logloss", random_state=rs, verbosity=0))]),
    }

## 2. Pooled Leave-One-Subject-Out

Mỗi fold giữ lại **một bệnh nhân** làm test, huấn luyện trên 59 người còn lại. Bộ lọc IQR loại cửa sổ ngoại lai chỉ được fit trên phần train của từng fold, để không rò rỉ thông tin từ bệnh nhân test. Siêu tham số cố định từ các vòng tuning trước, tránh nested tuning 60 fold.

In [5]:
def pooled_loso(pooled, feature_cols):
    X = pooled[feature_cols]; y = pooled["status"].to_numpy()
    groups = pooled["record_id"].to_numpy(); sources = pooled["source"].to_numpy()
    logo = LeaveOneGroupOut(); n_folds = logo.get_n_splits(groups=groups)
    rows = []
    for name in build_models():
        t0 = time.time()
        for fold_i, (tr, te) in enumerate(logo.split(X, y, groups)):
            keep = iqr_train_mask(X.iloc[tr]).to_numpy()
            X_tr, y_tr, s_tr = X.iloc[tr][keep], y[tr][keep], sources[tr][keep]
            pipe = build_models()[name]
            pipe.fit(X_tr, y_tr, clf__sample_weight=source_weights(s_tr))
            probs = pipe.predict_proba(X.iloc[te])[:, 1]
            for p, t in zip(probs, y[te]):
                rows.append({"record_id": groups[te][0], "source": sources[te][0], "status": int(t),
                             "model": name, "prob": float(p), "pred": int(p >= 0.5)})
        print(f"{name}: {n_folds} fold xong trong {time.time()-t0:.0f}s")
    return pd.DataFrame(rows)

def metrics_row(p, model, level, subset):
    y, yp, pr = p["status"], p["pred"], p["prob"]
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
    return {"Model": model, "Level": level, "Subset": subset,
            "Accuracy": accuracy_score(y, yp), "Recall": recall_score(y, yp, zero_division=0),
            "Specificity": tn / (tn + fp) if tn + fp else 0.0, "Precision": precision_score(y, yp, zero_division=0),
            "F1": f1_score(y, yp, zero_division=0), "ROC-AUC": roc_auc_score(y, pr), "FN": int(fn), "FP": int(fp)}

def summarize(pred):
    rows = []
    for model in pred["model"].unique():
        p = pred[pred["model"] == model]
        rows.append(metrics_row(p, model, "window", "pooled"))
        for src in ("mimic", "afdb"):
            rows.append(metrics_row(p[p["source"] == src], model, "window", src))
        pm = p[p["source"] == "mimic"]
        by = pm.groupby("record_id").agg(status=("status", "first"), prob=("prob", "mean")).reset_index()
        by["pred"] = (by["prob"] >= 0.5).astype(int)
        rows.append(metrics_row(by, model, "subject", "mimic"))   # mức bệnh nhân chỉ cho MIMIC
    return pd.DataFrame(rows)

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_name = "XGBoost"
if RUN_POOLED_LOSO:
    predictions = pooled_loso(pooled, FEATURE_COLS)
    predictions.to_csv(OUTPUT_DIR / "pooled_loso_predictions.csv", index=False)
    summary = summarize(predictions)
    summary.to_csv(OUTPUT_DIR / "pooled_loso_results.csv", index=False)
    pooled_rows = summary[(summary.Level == "window") & (summary.Subset == "pooled")]
    best_name = pooled_rows.sort_values("ROC-AUC", ascending=False).iloc[0]["Model"]
    display(summary.round(4))
print("Model được chọn:", best_name)

Logistic Regression: 60 fold xong trong 11s


Random Forest: 60 fold xong trong 91s


XGBoost: 60 fold xong trong 30s


,Model,Level,Subset,Accuracy,Recall,Specificity,Precision,F1,ROC-AUC,FN,FP
0,Logistic Regression,window,pooled,0.9479,0.9743,0.9300,0.9040,0.9378,0.9702,341,1372
1,Logistic Regression,window,mimic,0.9036,0.9706,0.8242,0.8676,0.9162,0.8678,66,332
2,Logistic Regression,window,afdb,0.9542,0.9750,0.9413,0.9118,0.9423,0.9812,275,1040
3,Logistic Regression,subject,mimic,0.9429,1.0000,0.8750,0.9048,0.9500,0.8750,0,2
4,Random Forest,window,pooled,0.9539,0.9649,0.9464,0.9241,0.9441,0.9880,465,1051
5,Random Forest,window,mimic,0.9215,0.9710,0.8628,0.8937,0.9307,0.9486,65,259
6,Random Forest,window,afdb,0.9585,0.9637,0.9553,0.9306,0.9469,0.9922,400,792
7,Random Forest,subject,mimic,0.9429,1.0000,0.8750,0.9048,0.9500,0.9375,0,2
8,XGBoost,window,pooled,0.9599,0.9695,0.9534,0.9337,0.9512,0.9881,405,913
9,XGBoost,window,mimic,0.9121,0.9746,0.8379,0.8772,0.9233,0.9302,57,306


Model được chọn: XGBoost


## 3. Huấn luyện model cuối trên toàn bộ 60 bệnh nhân và xuất file

Model cuối được fit trên tất cả dữ liệu (sau lọc IQR và cân bằng nguồn) rồi lưu cùng *model card* ghi phiên bản, danh sách đặc trưng theo đúng thứ tự, và kết quả LOSO. AI-Service đọc model này qua `app/models/v4.pkl`.

In [7]:
X = pooled[FEATURE_COLS]; y = pooled["status"].to_numpy()
keep = iqr_train_mask(X).to_numpy()
final = build_models()[best_name]
final.fit(X[keep], y[keep], clf__sample_weight=source_weights(pooled["source"].to_numpy()[keep]))

model_path = OUTPUT_DIR / "healthsense_afib_pipeline.pkl"
joblib.dump(final, model_path)
card = {
    "name": "HealthSense AFib Detector", "version": "4.1.0", "model": best_name,
    "trained_on": f"{pooled.record_id.nunique()} patients: MIMIC PERform AF (PPG 125Hz) + MIT-BIH AFDB (ECG)",
    "training_windows": int(keep.sum()), "features": FEATURE_COLS,
    "input": "Cửa sổ 30s: chuỗi NN (ms) -> 13 đặc trưng HRV theo thứ tự `features`",
    "output": "predict_proba[:, 1] = P(AFib); ngưỡng mặc định 0.5",
    "source_balancing": "sample weight — mỗi dataset đóng góp tổng trọng số bằng nhau",
}
if RUN_POOLED_LOSO:
    r = summary[(summary.Model == best_name) & (summary.Level == "window") & (summary.Subset == "pooled")].iloc[0]
    card["evaluation"] = {"pooled_loso_window": {k: round(float(r[k]), 4) for k in ("Accuracy", "Recall", "Specificity", "ROC-AUC")}}
json.dump(card, open(OUTPUT_DIR / "model_card.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Đã lưu:", model_path, f"({model_path.stat().st_size/1024:.0f} KB)")

Đã lưu: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab\models\final\healthsense_afib_pipeline.pkl (242 KB)
